In [1]:
import random

import numpy as np
import pandas as pd

from utils.config import Config
from data_handler import DataHandler
from classificators.dummy_classifier import DummyClassifier
from classificators.random_forest_classifier import RandomForestClassifierSK
from utils.utils import calculate_mcc_multilabel, plot_per_class_confusion


In [2]:
config = Config()

# Seeding
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
# if you use any other libraries that require seeding, set it here as well (e.g., torch.manual_seed(SEED) for PyTorch)
# -> your results should be reproducible across runs with the same seed


val_mccs = []
test_mccs = []
lr_histories_by_fold = {}

# load data
datahandler = DataHandler(config=config)


Local dataset found: data\cps_data_multi_label.pkl
Loading data into memory...
Data loaded.


In [ ]:

# Leave-one-out: EXPERIMENT_ID = 1..4
for fold in range(1, 5):
    print(f"\n--- Fold {fold}/4 | EXPERIMENT_ID={fold} ---")
    val_id = fold + 1 if fold < 4 else 1

    datahandler.config.data.test_experiment_id = fold
    # validation hat to be different from test
    datahandler.config.data.validation_experiment_id = val_id

    train, val, test, target_vals = datahandler.get_data_loaders()

    print(train.describe())


--- Fold 1/4 | EXPERIMENT_ID=1 ---
Starting data preparation...
             time     Acc.x     Acc.y     Acc.z    Gyro.x    Gyro.y    Gyro.z  \
126191    44.6168  0.063255 -0.212832 -0.258858  0.391883  0.109480  0.270023   
126192    44.6173  0.062836 -0.211914 -0.258330  0.391126  0.108949  0.272111   
126193    44.6179  0.063507 -0.210768 -0.258827  0.392641  0.110011  0.271980   
126194    44.6184  0.062752 -0.211169 -0.257926  0.392424  0.108596  0.271719   
126195    44.6190  0.062920 -0.210080 -0.257708  0.394697  0.109126  0.270545   
...           ...       ...       ...       ...       ...       ...       ...   
2069966  541.3810  0.070050 -0.207672 -0.253794  0.378788  0.083304  0.270284   
2069967  541.3815  0.070554 -0.208130 -0.253483  0.380303  0.084188  0.269371   
2069968  541.3820  0.070805 -0.208589 -0.253297  0.380411  0.083304  0.269502   
2069969  541.3825  0.070973 -0.209048 -0.252986  0.381277  0.082243  0.268197   
2069970  541.3830  0.071644 -0.209277 -0.252